## SNN: Sequence Similarity Network builder 
Metodología adaptada de Seo *et al*., 2025 para la construcción de una Sequence Similarity Network (SSN) basada en la similitud entre secuencias proteicas de PETasas procedentes de organismos extremófilos.

El análisis se realizó al conjunto completo de PETasas clasificadas como extremas, extremotolerantes y no extremas, con el objetivo de comparar su organización dentro de la red y evaluar posibles patrones de agrupamiento asociados a sus características ecológicas y moleculares.

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from ssnHeuristic import SSNBuilder

### 1. Preparación de archivos

Comenzamos cargando la tabla de PETasas:

In [ ]:
# Cargamos la tabla de PETasas
PETases = pd.read_excel("C:/Users/HP/OneDrive/Escritorio/Practicas_pLMs/Tabla_PETasas.xlsx", sheet_name="PETases")
PETases.head()

,qseqid,sseqid,pident,qlen,slen,length,qstart,qend,sstart,send,...,Latitude,Longitude,Isolation Source,true_extreme_environment,number_extr_environments,type_env_extr,environment,env comments,Final especie,Observations
0,GCA_913061175.1_SRR3933257_bin.1_MetaBAT_v2.12...,P0173,71.111,314,319,315,1,313,4,318,...,46.431000,-48.480000,marine metagenome,no,NaN,NaN,NaN,NaN,uncultured Psychrobacter sp.,NaN
1,AIK49165.1,P0111,81.768,214,205,181,34,214,25,205,...,55.939676,9.515585,Rhizosphere,no,NaN,NaN,NaN,NaN,Bacillus atrophaeus subsp. globigii,NaN
2,GCA_002345575.1_ASM234557v1_genomic_00658,P0176,73.733,218,218,217,1,217,1,217,...,40.680000,-73.910000,metal/plastic,no,NaN,NaN,NaN,NaN,Stutzerimonas stutzeri,NaN
3,GCA_002345575.1_ASM234557v1_genomic_03654,P0174,67.662,201,201,201,1,201,1,201,...,40.680000,-73.910000,metal/plastic,no,NaN,NaN,NaN,NaN,Stutzerimonas stutzeri,NaN
4,MBX7275218.1,P0176,84.332,218,218,217,1,217,1,217,...,28.613889,77.208889,ice melting water from glacier site,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,0.0


A continuación, creamos el archivo FASTA a partir de la columna de secuencias de la tabla de PETasas:

In [ ]:
# Creamos el FASTA para el total de PETasas 
with open("PETasas_totales_FASTA.fa", "w") as f:
    for i, row in PETases.iterrows():
        f.write(f">{row['qseqid']}\n")
        f.write(f"{row['query_seq']}\n")

print("Archivo FASTA creado correctamente")

Archivo FASTA creado correctamente


Seguidamente, lanzamos el archivo FASTA en Clustal Omega (https://www.ebi.ac.uk/jdispatcher/msa/clustalo) y descargamos el archivo correspondiente a "percentage identity matrix" (PIM) necesario para calcular la matriz de distancias. La matriz de distancias se calculó a partir del archivo PIM siguiendo la siguiente fórmula: 
Distancia = 1 - (Identidad / 100)

Este cálculo se hizo direntamente en la tabla Excel y se guardaron los resultados en otra hoja denominada "MD".

Cargamos la matriz de distancias:

In [ ]:
# Matriz de distancias PETases totales
matriz_distancias_totales = pd.read_excel("C:/Users/HP/OneDrive/Escritorio/Practicas_pLMs/PETasas_totales_pim.xlsx", sheet_name="MD", index_col=0)
matriz_distancias_totales.head()

,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,...,Unnamed: 1313,Unnamed: 1314,Unnamed: 1315,Unnamed: 1316,Unnamed: 1317,Unnamed: 1318,Unnamed: 1319,Unnamed: 1320,Unnamed: 1321,Unnamed: 1322
QHE53498.1,0.0000,0.2490,0.3105,0.2984,0.2886,0.2764,0.2996,0.2955,0.2834,0.4498,...,0.8125,0.8214,0.8304,0.8214,0.8482,0.8482,0.8393,0.8393,0.8482,0.8304
MCP8617662.1,0.2490,0.0000,0.3266,0.2984,0.3008,0.2967,0.2996,0.2996,0.2713,0.4458,...,0.7946,0.8125,0.8214,0.8125,0.8393,0.8393,0.8304,0.8214,0.8304,0.8125
MBP2256871.1,0.3105,0.3266,0.0000,0.2903,0.2927,0.2846,0.2874,0.2834,0.2996,0.4315,...,0.8125,0.8304,0.8393,0.8214,0.8393,0.8393,0.8393,0.8304,0.8482,0.8304
MCF3942252.1,0.2984,0.2984,0.2903,0.0000,0.2602,0.2602,0.2915,0.2874,0.2551,0.4355,...,0.8036,0.8214,0.8304,0.8125,0.8304,0.8393,0.8214,0.8214,0.8304,0.8125
BAC14385.1,0.2886,0.3008,0.2927,0.2602,0.0000,0.0610,0.2764,0.2520,0.2602,0.4472,...,0.8214,0.8304,0.8304,0.8125,0.8482,0.8482,0.8393,0.8393,0.8393,0.8214


Una vez tenemos estos archivos preparados, podemos comenzar con la construcción del Network

### 2. Incializamos SSNBuilder

In [ ]:
# Definimos una carpeta destino
archivo_fasta = "PETases_totales_FASTA.fa"
output_folder = "SNN"

In [ ]:
# Inicializamos SSNBiulder
ssn_total = SSNBuilder(archivo_fasta, output_folder)

In [ ]:
# Añadimos las distancias al SSNBuilder
df_total = matriz_distancias_totales
ssn_total.df = df_total

### 3. Calculamos los clústers y construimos el Network

In [ ]:
# Analizamos todas las secuencias con los hiperparámetros predeterminados
results_total= ssn_total.all_results = {seq: ssn_total.analyze_seq(seq) for seq in df_total.index}

# Construimos el grafo NetworkX
ssn_total.buildNetwork(out_folder="ssn_heuristic_test_total/")

# Score del Network (útil para la optimización de parámetros)
G_ssn_total = ssn_total.getGraph()
score_total = SSNBuilder.score_ssn(G_ssn_total)

score_total

Net file written: ssn_heuristic_test_total_27ABR26/net_single.net
Edges: 10610, Nodes: 1322


-13.746535796766743

### 4. Optimización de los Hiperparámetros

Hacemos una optimización de los hiperparámetros para recalcular los valores de r y así asegurar la calidad del SSN:

In [ ]:
r1_vals = np.linspace(5e-4, 2e-3, 5)
r2_vals = np.linspace(1e-3, 6e-3, 5)
r3_vals = np.linspace(1e-2, 5e-2, 5)

best_score = -np.inf
best_params = None

for r1 in r1_vals:
    for r2 in r2_vals:
        for r3 in r3_vals:
            if not (r1 < r2 < r3):
                continue

            # Reseteamos los valores previos
            ssn_total.all_results = {}

            # Ejecutar análisis de histogramas por secuencia
            for seq in ssn_total.df.index:
                ssn_total.analyze_seq(seq, r1=r1, r2=r2, r3=r3)

            # Contrucción del grafo a partir de todos los resultados
            G_total = ssn_total.getGraph()

            # Puntuación del grafo
            s = SSNBuilder.score_ssn(G_total)

            if s > best_score:
                best_score = s
                best_params = (r1, r2, r3)

print("Optimal r1,r2,r3:", best_params, "Score:", best_score)

Optimal r1,r2,r3: (0.000875, 0.001, 0.01) Score: -13.746535796766743


Repetimos el Network con los nuevos valores optimizados de r:

In [ ]:
# Analizamos todas las secuencias con los hiperparámetros predeterminados
results_total = ssn_total.all_results = {seq: ssn_total.analyze_seq(seq , r1=0.000875, r2=0.001, r3=0.01) for seq in df_total.index}

# Construimos el grafo NetworkX
ssn_total.buildNetwork(out_folder="ssn_heuristic_test_total/")

# Score del Network 
G_ssn_total = ssn_total.getGraph()
score_total = SSNBuilder.score_ssn(G_ssn_total)

score_total

Net file written: ssn_heuristic_test_total_27ABR26/net_single.net
Edges: 10610, Nodes: 1322


-13.746535796766743

**En la carpeta "ssn_heuristic_test_total/" se encuentra el Network construido, el cual visualizaremos mediante Cytoscape (v3.10.4)**